In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

RAW_KG_PATH = "kg.csv"

pd.set_option('display.max_rows', None)  # 行数无上限
pd.set_option('display.max_colwidth', None) # 单元格宽度无上限

def batch_inspect_ontology_nacc():
    print("==================================================")
    print("🛠️ 步骤 1: 启动 PrimeKG 本体对齐批量检修工具 (NACC专版)...")
    
    try:
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
        print(f"   ✅ 成功加载 {RAW_KG_PATH}。")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {RAW_KG_PATH}，请检查路径。")
        return

    print("\n==================================================")
    print("🧹 步骤 2: 划定安全检索范围...")
    target_types = ['disease', 'effect/phenotype']
    search_df = df[df['x_type'].isin(target_types)]
    print(f"   已将检索范围锁定在 {len(search_df)} 条 'disease' 和 'effect/phenotype' 关系中。")
    
    search_strategy = {
        "核心-轻度认知障碍 (MCI)": ["mild cognitive impairment", "cognitive decline"],
        "认知-记忆/失语/执行 (Memory/Language)": ["memory impairment", "amnesia", "aphasia", "executive functioning"],
        "精神-创伤/双相/强迫 (PTSD/Bipolar/OCD)": ["post-traumatic", "bipolar", "obsessive-compulsive", "schizophrenia"],
        "病史-心衰/房颤/脑缺血 (CHF/AFIB/TIA)": ["heart failure", "atrial fibrillation", "ischemic attack", "transient ischemic"],
        "病史-脑外伤/癫痫 (TBI/Seizure)": ["traumatic brain injury", "seizure", "epilepsy"],
        "病史-代谢/内分泌 (Diabetes/Thyroid)": ["diabetes mellitus", "thyroid", "vitamin b12"]
    }

    print("\n==================================================")
    print("🔍 步骤 3: 开始按医学词根进行批量模糊检索...")
    
    for category, keywords in search_strategy.items():
        print(f"\n▶ 正在检索【{category}】...")
        print(f"   使用的词根: {keywords}")
        
        pattern = '|'.join(keywords)
        mask = search_df['x_name'].str.contains(pattern, case=False, na=False)
        results = search_df[mask]['x_name'].unique()
        
        if len(results) > 0:
            print(f"   ✅ 找到 {len(results)} 个候选节点，前 10 个最精准的候选如下：")
            sorted_results = sorted(results, key=len)
            for i, res in enumerate(sorted_results[:10]):
                print(f"      [{i+1}] {res}")
        else:
            print(f"   ❌ 使用这些词根未能找到任何节点。")

    print("\n==================================================")
    print("💡 步骤 4: 检修结果输出完毕。")
    print("==================================================")

if __name__ == "__main__":
    batch_inspect_ontology_nacc()

🛠️ 步骤 1: 启动 PrimeKG 本体对齐批量检修工具 (NACC专版)...
   ✅ 成功加载 kg.csv。

🧹 步骤 2: 划定安全检索范围...
   已将检索范围锁定在 598340 条 'disease' 和 'effect/phenotype' 关系中。

🔍 步骤 3: 开始按医学词根进行批量模糊检索...

▶ 正在检索【核心-轻度认知障碍 (MCI)】...
   使用的词根: ['mild cognitive impairment', 'cognitive decline']
   ✅ 找到 1 个候选节点，前 10 个最精准的候选如下：
      [1] neurodegeneration, childhood-onset, with ataxia, tremor, optic atrophy, and cognitive decline

▶ 正在检索【认知-记忆/失语/执行 (Memory/Language)】...
   使用的词根: ['memory impairment', 'amnesia', 'aphasia', 'executive functioning']
   ✅ 找到 31 个候选节点，前 10 个最精准的候选如下：
      [1] Aphasia
      [2] aphasia
      [3] Motor aphasia
      [4] nominal aphasia
      [5] Memory impairment
      [6] retrograde amnesia
      [7] anterograde amnesia
      [8] dissociative amnesia
      [9] Long term memory impairment
      [10] primary progressive aphasia

▶ 正在检索【精神-创伤/双相/强迫 (PTSD/Bipolar/OCD)】...
   使用的词根: ['post-traumatic', 'bipolar', 'obsessive-compulsive', 'schizophrenia']
   ✅ 找到 12 个候选节点，前 10 个最精准的候选如下：
      [1] schiz